# Validation MuQ FP32 + MusicFM palette6 + head palette6

Menghitung MuQ dari audio dan memakai ulang fitur MusicFM palette6 yang sudah divalidasi. Anotasi, boundary, dan head tetap. Cache baru terpisah; proses dapat dilanjutkan dengan memakai kembali nilai `output` dan `cache` dari sesi sebelumnya. Membutuhkan beberapa GiB cache tambahan dan macOS.

Jalankan dari root Section-Annotations, kernel `.venv-songformer311`. Tahap ini belum melakukan kompresi MuQ atau integrasi aplikasi.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys, json
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT = Path('/Users/danendra/Documents/ADA26/C6/SongMap')
# Bisa diubah ke path report tertentu jika ada beberapa eksperimen.
exports = sorted(Path('outputs.local').glob('muq-fp32-*/export_report.json'), key=lambda p:p.stat().st_mtime, reverse=True)
assert exports, 'Jalankan notebook ekspor MuQ FP32 dahulu'
proof = exports[0]
candidates=[]
for path in Path('outputs.local').glob('musicfm-device-combo-*/*.json'):
    try:
        r=json.loads(path.read_text())
        if (r.get('full_validation') and r.get('head_scheme')=='palette6'
            and (r.get('musicfm_override') or {}).get('compression_scheme')=='palette6'):
            candidates.append(path)
    except (ValueError, OSError):
        pass
assert candidates, 'Report kombinasi MusicFM palette6 + head palette6 belum ditemukan; isi baseline secara manual'
baseline=max(candidates,key=lambda p:p.stat().st_mtime)
output=Path('outputs.local')/('muq-validation-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
cache=output/'cache'
print('Python:',sys.executable)
print('MuQ source:',proof)
print('Baseline:',baseline)
print('Output:',output)


In [ ]:
def run_validation(limit,name):
    output.mkdir(parents=True,exist_ok=True)
    report=output/(name+'.json')
    log=output/(name+'.log')
    if report.exists():
        print('Report sudah tersedia:',report)
        return
    command=[sys.executable,'-u','-m','deployment.evaluate_muq',
        '--songmap-root',str(SONGMAP_ROOT),'--baseline-combination',str(baseline),
        '--muq-export-report',str(proof),'--cache-dir',str(cache),
        '--limit',str(limit),'--output',str(report)]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Proses gagal ({status}); lihat {log.resolve()}')
run_validation(1,'one-validation')


In [ ]:
# Memakai ulang lagu yang sudah selesai dari smoke run.
run_validation(0,'all-validation')
r=json.loads((output/'all-validation.json').read_text())
print('Tracks:',r['tracks'],'| Full validation:',r['full_validation'])
print('Baseline F1:',r['baseline_metrics']['macro_f1_present'])
print('MuQ FP32 F1:',r['metrics']['macro_f1_present'])
print('Accuracy:',r['metrics']['section_accuracy'])
print('F1 delta:',r['macro_f1_delta'])
print('Changed sections:',len(r['changed_sections']))
print('Per-class:',json.dumps(r['metrics']['per_class'],indent=2))
print('Report:',(output/'all-validation.json').resolve())


Bagikan `all-validation.json`. Nilai delta membandingkan MuQ FP32 baru dengan MuQ aplikasi lama, memakai MusicFM palette6 dan head palette6 yang sama pada subset lagu yang sama. Ini evaluasi label dengan boundary Apple tetap, bukan akurasi deteksi boundary atau benchmark iPhone. Setelah baseline ini diperiksa, baru kompres MuQ INT8/palette6 dari sumber FP32 tersebut.